In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import praw
import os
import json
from datetime import datetime

In [4]:
# Initialize Reddit API client
reddit = praw.Reddit(
    client_id=os.getenv('REDDIT_CLIENT_ID'),
    client_secret=os.getenv('REDDIT_SECRET'),
    user_agent='idea_finder_agent:v1.0 (by u/your_username)',  # Change this to your username
)

print(f"Reddit instance created. Read-only mode: {reddit.read_only}")
print(f"Client ID: {os.getenv('REDDIT_CLIENT_ID')[:10]}...")  # Show first 10 chars for verification

Reddit instance created. Read-only mode: True
Client ID: 7RqfB6RqZG...


In [5]:
def search_reddit_posts(query, subreddit_name="all", sort="relevance", time_filter="all", limit=25):
    """
    Search Reddit posts using PRAW
    
    Args:
        query (str): Search query (e.g., "saas ideas")
        subreddit_name (str): Subreddit to search in (default: "all")
        sort (str): Sort method - "relevance", "hot", "top", "new", "comments"
        time_filter (str): Time filter - "all", "day", "hour", "month", "week", "year"
        limit (int): Number of results to return (default: 25)
    
    Returns:
        list: List of dictionaries containing post information
    """
    try:
        # Get the subreddit
        subreddit = reddit.subreddit(subreddit_name)
        
        # Search for posts
        search_results = subreddit.search(
            query=query,
            sort=sort,
            time_filter=time_filter,
            limit=limit
        )
        
        posts = []
        for submission in search_results:
            post_data = {
                'title': submission.title,
                'author': str(submission.author) if submission.author else '[deleted]',
                'score': submission.score,
                'upvote_ratio': submission.upvote_ratio,
                'num_comments': submission.num_comments,
                'created_utc': datetime.fromtimestamp(submission.created_utc),
                'subreddit': str(submission.subreddit),
                'url': submission.url,
                'permalink': f"https://reddit.com{submission.permalink}",
                'selftext': submission.selftext[:200] + "..." if len(submission.selftext) > 200 else submission.selftext,
                'is_self': submission.is_self,
                'id': submission.id
            }
            posts.append(post_data)
        
        return posts
    
    except Exception as e:
        print(f"Error searching Reddit: {e}")
        return []

In [6]:
# Example: Search for SaaS ideas
print("Searching for 'saas ideas' on Reddit...")
saas_posts = search_reddit_posts(
    query="saas ideas",
    subreddit_name="entrepreneur+startups+SaaS",  # Search multiple relevant subreddits
    sort="relevance",
    time_filter="month",  # Posts from the last month
    limit=10
)

print(f"Found {len(saas_posts)} posts")
print("\n" + "="*80)

# Display results
for i, post in enumerate(saas_posts, 1):
    print(f"\n{i}. {post['title']}")
    print(f"   Author: u/{post['author']} | Subreddit: r/{post['subreddit']}")
    print(f"   Score: {post['score']} | Comments: {post['num_comments']} | Date: {post['created_utc'].strftime('%Y-%m-%d')}")
    if post['selftext']:
        print(f"   Text: {post['selftext']}")
    print(f"   Link: {post['permalink']}")
    print("-" * 60)

Searching for 'saas ideas' on Reddit...
Found 10 posts


1. SaaS Ideas?
   Author: u/I-Gone-Mad | Subreddit: r/SaaS
   Score: 11 | Comments: 31 | Date: 2025-08-31
   Text: How to find good ideas for a Web based SaaS ?

ANY RECOMMENDATIONS ? 
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/
------------------------------------------------------------

2. Looking for Micro-SaaS ideas to build or improve
   Author: u/Ielbdev | Subreddit: r/SaaS
   Score: 4 | Comments: 3 | Date: 2025-08-21
   Text: I’m a software engineer building SaaS products. I’m looking for a micro-SaaS idea that solves a focused problem or an existing SaaS that could be improved.

What I’m interested in:

* Tools with payin...
   Link: https://reddit.com/r/SaaS/comments/1mwcvkr/looking_for_microsaas_ideas_to_build_or_improve/
------------------------------------------------------------

3. How do you guys brainstorm ideas to build SaaS?
   Author: u/BlazingBrushes | Subreddit: r/SaaS
   Score: 9 | Comme

In [6]:
# Advanced search examples

def search_and_save_results(queries, filename="reddit_search_results.json"):
    """
    Search for multiple queries and save results to a JSON file
    """
    all_results = {}
    
    for query in queries:
        print(f"Searching for: '{query}'...")
        results = search_reddit_posts(
            query=query,
            subreddit_name="entrepreneur+startups+SaaS+business+sideproject",
            sort="top",
            time_filter="month",
            limit=15
        )
        all_results[query] = results
        print(f"Found {len(results)} posts for '{query}'")
    
    # Save to JSON file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to {filename}")
    return all_results

# Example queries for business ideas
idea_queries = [
    "saas ideas",
    "startup ideas 2024",
    "micro saas",
    "business ideas",
    "profitable startup ideas",
    "small business ideas",
    "side project ideas"
]

# Uncomment the line below to run the search
# results = search_and_save_results(idea_queries)

In [7]:
# Utility functions for analyzing search results

def filter_high_quality_posts(posts, min_score=5, min_comments=2):
    """
    Filter posts based on engagement metrics
    """
    return [post for post in posts if post['score'] >= min_score and post['num_comments'] >= min_comments]

def analyze_search_results(posts):
    """
    Analyze search results and provide insights
    """
    if not posts:
        return "No posts to analyze"
    
    total_posts = len(posts)
    avg_score = sum(post['score'] for post in posts) / total_posts
    avg_comments = sum(post['num_comments'] for post in posts) / total_posts
    
    # Top subreddits
    subreddit_counts = {}
    for post in posts:
        subreddit = post['subreddit']
        subreddit_counts[subreddit] = subreddit_counts.get(subreddit, 0) + 1
    
    top_subreddits = sorted(subreddit_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    analysis = f"""
    📊 SEARCH RESULTS ANALYSIS
    {'='*50}
    Total posts found: {total_posts}
    Average score: {avg_score:.1f}
    Average comments: {avg_comments:.1f}
    
    📍 Top Subreddits:
    """
    
    for subreddit, count in top_subreddits:
        analysis += f"\n    r/{subreddit}: {count} posts"
    
    return analysis

# Example usage functions
def search_specific_subreddit(query, subreddit):
    """
    Search within a specific subreddit
    """
    print(f"Searching r/{subreddit} for '{query}'...")
    return search_reddit_posts(query, subreddit, limit=10)

# Quick search function for common business idea searches
def quick_idea_search(idea_type="saas"):
    """
    Quick search for different types of business ideas
    """
    search_terms = {
        "saas": "saas ideas OR micro saas OR software ideas",
        "ecommerce": "ecommerce ideas OR online store ideas",
        "service": "service business ideas OR consulting ideas",
        "app": "app ideas OR mobile app startup",
        "ai": "ai startup ideas OR machine learning business"
    }
    
    query = search_terms.get(idea_type.lower(), idea_type)
    return search_reddit_posts(query, "entrepreneur+startups+business", limit=20)

In [8]:
# Analyze the SaaS search results
print(analyze_search_results(saas_posts))

# Filter for high-quality posts
high_quality_posts = filter_high_quality_posts(saas_posts, min_score=8, min_comments=20)
print(f"\n🎯 HIGH QUALITY POSTS (Score ≥8, Comments ≥20): {len(high_quality_posts)}")

for i, post in enumerate(high_quality_posts, 1):
    print(f"\n{i}. {post['title']}")
    print(f"   Score: {post['score']} | Comments: {post['num_comments']}")
    print(f"   Link: {post['permalink']}")

# Example of searching a specific subreddit
print("\n" + "="*80)
print("🔍 SEARCHING SPECIFIC SUBREDDIT: r/entrepreneur")
entrepreneur_posts = search_specific_subreddit("startup ideas", "entrepreneur")
print(f"Found {len(entrepreneur_posts)} posts in r/entrepreneur")


    📊 SEARCH RESULTS ANALYSIS
    Total posts found: 10
    Average score: 7.9
    Average comments: 24.0

    📍 Top Subreddits:
    
    r/SaaS: 9 posts
    r/Entrepreneur: 1 posts

🎯 HIGH QUALITY POSTS (Score ≥8, Comments ≥20): 5

1. SaaS Ideas?
   Score: 11 | Comments: 31
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/

2. How do you guys brainstorm ideas to build SaaS?
   Score: 8 | Comments: 28
   Link: https://reddit.com/r/SaaS/comments/1namong/how_do_you_guys_brainstorm_ideas_to_build_saas/

3. How to Validate Your SaaS Idea Before Launching
   Score: 23 | Comments: 37
   Link: https://reddit.com/r/SaaS/comments/1nd6nac/how_to_validate_your_saas_idea_before_launching/

4. I'm getting fired so I'm gonna focus on building my SaaS idea
   Score: 10 | Comments: 22
   Link: https://reddit.com/r/SaaS/comments/1n8zef9/im_getting_fired_so_im_gonna_focus_on_building_my/

5. I'm a developer with a SaaS idea. How do I get my first customers before I start building?
   Scor

In [7]:
def fetch_post_comments(post_id, limit_comments=10, limit_more=5):
    """
    Fetch comments for a specific Reddit post
    
    Args:
        post_id (str): Reddit post ID
        limit_comments (int): Maximum number of comments to return (0 for all)
        limit_more (int): Limit for MoreComments replacement (0 to remove all, None for all)
    
    Returns:
        list: List of comment dictionaries
    """
    try:
        # Get the submission by ID
        submission = reddit.submission(id=post_id)
        
        # Replace MoreComments objects with actual comments
        # limit=5 means replace up to 5 "load more comments" sections
        # This balances between getting enough comments and API rate limits
        submission.comments.replace_more(limit=limit_more)
        
        # Get all comments as a flat list
        all_comments = submission.comments.list()
        
        comments_data = []
        for i, comment in enumerate(all_comments):
            if limit_comments > 0 and i >= limit_comments:
                break
                
            comment_data = {
                'id': comment.id,
                'body': comment.body,
                'author': str(comment.author) if comment.author else '[deleted]',
                'score': comment.score,
                'created_utc': datetime.fromtimestamp(comment.created_utc),
                'parent_id': comment.parent_id,
                'is_submitter': comment.is_submitter,
                'depth': comment.depth if hasattr(comment, 'depth') else 0,
                'permalink': f"https://reddit.com{comment.permalink}"
            }
            comments_data.append(comment_data)
        
        return comments_data
    
    except Exception as e:
        print(f"Error fetching comments for post {post_id}: {e}")
        return []

# Test the function with one of our SaaS posts
if saas_posts:
    test_post = saas_posts[0]  # Get the first post
    print(f"Testing comment fetching for post: '{test_post['title']}'")
    print(f"Post ID: {test_post['id']}")
    print(f"Original comment count: {test_post['num_comments']}")
    
    # Fetch up to 5 comments to test
    comments = fetch_post_comments(test_post['id'], limit_comments=5, limit_more=2)
    
    print(f"\nFetched {len(comments)} comments:")
    print("="*80)
    
    for i, comment in enumerate(comments, 1):
        print(f"\n{i}. Comment by u/{comment['author']} (Score: {comment['score']})")
        print(f"   Posted: {comment['created_utc'].strftime('%Y-%m-%d %H:%M')}")
        print(f"   Body: {comment['body'][:150]}{'...' if len(comment['body']) > 150 else ''}")
        print(f"   Link: {comment['permalink']}")
        print("-" * 60)
else:
    print("No SaaS posts available for testing. Run the search cells first.")

Testing comment fetching for post: 'SaaS Ideas?'
Post ID: 1n5chir
Original comment count: 31

Fetched 5 comments:

1. Comment by u/Junior_Bid_6652 (Score: 6)
   Posted: 2025-08-31 20:53
   Body: The best place to find SaaS ideas is by looking for people's complaints.

Reddit comments are a goldmine of unmet needs. People often vent about their...
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/nbrurv5/
------------------------------------------------------------

2. Comment by u/Electronic-Cause5274 (Score: 3)
   Posted: 2025-08-31 22:34
   Body: Some of the best SaaS ideas come from scratching your own itch. Beyond Reddit rants, look at industries where processes still run on spreadsheets or e...
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/nbs8xap/
------------------------------------------------------------

3. Comment by u/CaffeinatedTech (Score: 2)
   Posted: 2025-08-31 22:47
   Body: Local businesses.
   Link: https://reddit.com/r/SaaS/comments/1n

In [8]:
def search_reddit_posts_with_comments(query, subreddit_name="all", sort="relevance", time_filter="all", limit=25, 
                                      include_comments=True, max_comments=5, comment_limit_more=2):
    """
    Enhanced search function that includes comments for each post
    
    Args:
        query (str): Search query (e.g., "saas ideas")
        subreddit_name (str): Subreddit to search in (default: "all")
        sort (str): Sort method - "relevance", "hot", "top", "new", "comments"
        time_filter (str): Time filter - "all", "day", "hour", "month", "week", "year"
        limit (int): Number of results to return (default: 25)
        include_comments (bool): Whether to fetch comments for each post
        max_comments (int): Maximum number of comments to fetch per post (0 for all)
        comment_limit_more (int): Limit for MoreComments replacement
    
    Returns:
        list: List of dictionaries containing post information with comments
    """
    try:
        # Get the subreddit
        subreddit = reddit.subreddit(subreddit_name)
        
        # Search for posts
        search_results = subreddit.search(
            query=query,
            sort=sort,
            time_filter=time_filter,
            limit=limit
        )
        
        posts = []
        for submission in search_results:
            post_data = {
                'title': submission.title,
                'author': str(submission.author) if submission.author else '[deleted]',
                'score': submission.score,
                'upvote_ratio': submission.upvote_ratio,
                'num_comments': submission.num_comments,
                'created_utc': datetime.fromtimestamp(submission.created_utc),
                'subreddit': str(submission.subreddit),
                'url': submission.url,
                'permalink': f"https://reddit.com{submission.permalink}",
                'selftext': submission.selftext[:200] + "..." if len(submission.selftext) > 200 else submission.selftext,
                'is_self': submission.is_self,
                'id': submission.id,
                'comments': []  # Initialize empty comments list
            }
            
            # Fetch comments if requested
            if include_comments:
                print(f"Fetching comments for: {submission.title[:50]}...")
                try:
                    # Replace MoreComments objects
                    submission.comments.replace_more(limit=comment_limit_more)
                    
                    # Get comments as flat list
                    all_comments = submission.comments.list()
                    
                    # Limit comments if specified
                    comments_to_process = all_comments[:max_comments] if max_comments > 0 else all_comments
                    
                    for comment in comments_to_process:
                        comment_data = {
                            'id': comment.id,
                            'body': comment.body,
                            'author': str(comment.author) if comment.author else '[deleted]',
                            'score': comment.score,
                            'created_utc': datetime.fromtimestamp(comment.created_utc),
                            'parent_id': comment.parent_id,
                            'is_submitter': comment.is_submitter,
                            'depth': getattr(comment, 'depth', 0),
                            'permalink': f"https://reddit.com{comment.permalink}"
                        }
                        post_data['comments'].append(comment_data)
                
                except Exception as e:
                    print(f"  Error fetching comments: {e}")
                    post_data['comments'] = []
            
            posts.append(post_data)
        
        return posts
    
    except Exception as e:
        print(f"Error searching Reddit: {e}")
        return []

# Test the enhanced function
print("🧪 Testing enhanced search with comments...")
test_posts = search_reddit_posts_with_comments(
    query="micro saas", 
    subreddit_name="SaaS", 
    limit=2,  # Just 2 posts for testing
    max_comments=3,  # Just 3 comments per post
    comment_limit_more=1
)

print(f"\n✅ Retrieved {len(test_posts)} posts with comments")

# Display the results
for i, post in enumerate(test_posts, 1):
    print(f"\n{'='*80}")
    print(f"POST {i}: {post['title']}")
    print(f"Author: u/{post['author']} | Score: {post['score']} | Comments: {post['num_comments']}")
    print(f"Link: {post['permalink']}")
    
    if post['selftext']:
        print(f"Text: {post['selftext']}")
    
    print(f"\n📝 COMMENTS ({len(post['comments'])} fetched):")
    print("-" * 60)
    
    for j, comment in enumerate(post['comments'], 1):
        print(f"\n  {j}. u/{comment['author']} (Score: {comment['score']})")
        print(f"     {comment['body'][:100]}{'...' if len(comment['body']) > 100 else ''}")
    
    if not post['comments']:
        print("  No comments fetched.")
    
    print("\n" + "="*80)

🧪 Testing enhanced search with comments...
Fetching comments for: What are some small, focused micro-SaaS ideas that...
Fetching comments for: I will buy your micro SaaS...

✅ Retrieved 2 posts with comments

POST 1: What are some small, focused micro-SaaS ideas that you'd actually use or pay for?
Author: u/HalfOctober | Score: 8 | Comments: 11
Link: https://reddit.com/r/SaaS/comments/1m7etsr/what_are_some_small_focused_microsaas_ideas_that/
Text: Hey folks 👋

I’m exploring ideas to build a micro-SaaS product — something lightweight, super specific, and ideally something that solves a *real annoyance* or *niche problem* really well.

I’m especi...

📝 COMMENTS (3 fetched):
------------------------------------------------------------

  1. u/No_Molasses_1518 (Score: 5)
     I would 100% pay for a micro-SaaS that lets me bulk-check SaaS pricing pages for changes…especially ...

  2. u/[deleted] (Score: 3)
     [removed]

  3. u/Key-Boat-7519 (Score: 2)
     I'd pay for micro tools that ki

In [9]:
# Test the updated RedditIdeaFinder class from the script
import sys
sys.path.append('/home/mammalofski/projects/idea_finder_agent/src')

from reddit_idea_finder import RedditIdeaFinder

# Create an instance and test
print("🧪 Testing the updated RedditIdeaFinder class...")
finder = RedditIdeaFinder()

# Test with a simple search including comments
print("\n🔍 Testing search with comments...")
test_results = finder.search_posts(
    query="startup idea validation",
    subreddit_name="entrepreneur",
    limit=1,  # Just one post for testing
    include_comments=True,
    max_comments=2,
    comment_limit_more=1
)

print(f"✅ Found {len(test_results)} posts")

# Display using the new display method
finder.display_posts(test_results, "Test Results with Comments", show_comments=True)

🧪 Testing the updated RedditIdeaFinder class...
Reddit instance created. Read-only mode: True

🔍 Testing search with comments...
Fetching comments for: I spent $47k and 18 months building an "AI startup...
✅ Found 1 posts

Test Results with Comments

1. I spent $47k and 18 months building an "AI startup." Here's the brutal truth about why 90% of AI businesses are doomed.
   Author: u/Nipurn_1234 | Subreddit: r/Entrepreneur
   Score: 1644 | Comments: 503 | Date: 2025-08-06
   Text: **TL;DR:** Burned through $47k building an AI tool that 12 people use. Here's what the "AI gold rush" really looks like from the trenches, and why most AI startups are just expensive tech demos.

# The Setup (AKA How I Got Caught Up in the Hype)

18 months ago, I was a perfectly happy software consu...
   Link: https://reddit.com/r/Entrepreneur/comments/1mj1olw/i_spent_47k_and_18_months_building_an_ai_startup/

   💬 TOP COMMENTS (2 fetched):
      1. u/AutoModerator (Score: 1)
         Welcome to /r/Entrepren

In [10]:
# 🎯 FINAL DEMONSTRATION: Complete Reddit Idea Finder with Comments
print("🚀 COMPLETE REDDIT IDEA FINDER WITH COMMENTS")
print("="*60)

# Create a fresh instance
idea_finder = RedditIdeaFinder()

# 1. Basic search (no comments for speed)
print("\n1️⃣ BASIC SEARCH (No Comments)")
basic_results = idea_finder.search_posts(
    query="profitable business ideas", 
    subreddit_name="entrepreneur", 
    limit=3,
    include_comments=False
)
print(f"Found {len(basic_results)} posts")

# 2. Enhanced search WITH comments
print("\n2️⃣ ENHANCED SEARCH (With Comments)")
enhanced_results = idea_finder.search_posts(
    query="saas validation", 
    subreddit_name="entrepreneur+SaaS", 
    limit=2,
    include_comments=True,
    max_comments=2,
    comment_limit_more=1
)

# Display with comments
idea_finder.display_posts(enhanced_results, "SaaS Validation Ideas with Comments", show_comments=True)

# 3. Quick search for AI ideas with comments
print("\n3️⃣ QUICK AI SEARCH (With Comments)")
ai_results = idea_finder.quick_idea_search("ai", include_comments=True, max_comments=1)
idea_finder.display_posts(ai_results[:1], "AI Ideas with Comments", show_comments=True)

# 4. Analysis
print("\n4️⃣ ANALYSIS")
print(idea_finder.analyze_results(basic_results + enhanced_results))

print("\n✅ ALL FEATURES WORKING!")
print("💡 Key capabilities:")
print("   • Search Reddit posts with customizable parameters")
print("   • Fetch and display comments for each post")
print("   • Filter high-quality posts by engagement")
print("   • Analyze results and extract insights")
print("   • Save results to JSON files")
print("   • Quick searches for different business idea types")
print("   • Rate limit management for API calls")

🚀 COMPLETE REDDIT IDEA FINDER WITH COMMENTS
Reddit instance created. Read-only mode: True

1️⃣ BASIC SEARCH (No Comments)
Found 3 posts

2️⃣ ENHANCED SEARCH (With Comments)
Fetching comments for: How do you validate a saas idea?...
Fetching comments for: How to Validate Your SaaS Idea Before Launching...

SaaS Validation Ideas with Comments

1. How do you validate a saas idea?
   Author: u/zertour | Subreddit: r/SaaS
   Score: 3 | Comments: 23 | Date: 2025-06-14
   Text: I'm exploring some ideas in the construction management industry, but I don't know any decision makers at these types of companies. What are some strategies you've used that have worked in getting you in front of your market? How can I get to know the 'right' people?

\- LinkedIn?  
\- Subreddits?  ...
   Link: https://reddit.com/r/SaaS/comments/1lbmdyq/how_do_you_validate_a_saas_idea/

   💬 TOP COMMENTS (2 fetched):
      1. u/dOdrel (Score: 2)
         depends a lot on what you think the "right" people are. what has 